# 09 — Similar-projects recommender

Ports and **finishes** `notebooks_original/recommender-system.ipynb`. That
notebook built three cosine-similarity matrices over the apartment *projects* in
`appartments.csv` and then left several contradictory weighted sums in place
(`30*sim1 + 20*sim2 + 8*sim3` in one cell, `6*sim1 + 5*sim2 + 3*sim3` in
another) with no conclusion.

**Scope.** This is a *"which developments are like this one"* feature
(item-to-item over 246 projects). The preference / budget / bedroom recommender
over individual listings is a separate, unbuilt component — see
`PROJECT_PLAN.md` §10.

**What this notebook does.** All logic lives in `src/recommender/`; this is the
thin runner. It shows (1) that the three matrices are on different scales,
(2) the chosen blend `structural 0.5 / location 0.3 / facilities 0.2` on
min-max-normalised matrices, (3) a sanity check on three known projects, and
(4) weight sensitivity. Full derivation: `reports/recommender/blend_weights.md`.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from src.recommender.similarity import DEFAULT_WEIGHTS, build_components, load_appartments
from src.recommender.recommender import SimilarProjectsRecommender

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

raw = load_appartments()
raw_by_name = raw.set_index("PropertyName")
print(f"{len(raw)} projects   default weights: {DEFAULT_WEIGHTS}")

246 projects   default weights: {'structural': 0.5, 'location': 0.3, 'facilities': 0.2}


## 1. The three matrices are not on comparable scales

Cosine similarity is bounded, but the axes differ enough in spread and sign that
a naive weighted sum is dominated by `structural` on variance alone. This is why
each matrix is min-max normalised to [0, 1] before blending.

In [2]:
components = build_components(raw)

def offdiag_stats(m):
    n = m.shape[0]
    o = m[~np.eye(n, dtype=bool)]
    return {
        "min": o.min(), "max": o.max(), "mean": o.mean(), "std": o.std(),
        "pct_negative": (o < 0).mean(),
        "mean_top5_per_row": np.sort(m - np.eye(n) * 2, axis=1)[:, -5:].mean(),
    }

pd.DataFrame({k: offdiag_stats(v) for k, v in components.items()}).T.round(3)

,min,max,mean,std,pct_negative,mean_top5_per_row
facilities,0.000,0.683,0.072,0.082,0.000,0.365
structural,-0.766,1.000,0.033,0.341,0.546,0.855
location,-0.102,1.000,0.023,0.177,0.788,0.282


## 2. Sanity check — three known projects

`recommend()` returns the blended score plus the per-axis normalised similarity
for each pair, so the ranking is explainable.

In [3]:
rec = SimilarProjectsRecommender.from_csv()  # default 0.5 / 0.3 / 0.2

TARGETS = ["DLF The Arbour", "M3M Golf Hills", "Ireo Victory Valley"]
for t in TARGETS:
    print("=" * 90)
    print(f"QUERY: {t}  |  {raw_by_name.loc[t, 'PropertySubName']}")
    out = rec.recommend(t, k=5)
    for _, r in out.iterrows():
        sub = raw_by_name.loc[r["PropertyName"], "PropertySubName"]
        print(f"  {r['rank']}. {r['PropertyName']:<32} score={r['score']:.3f}  "
              f"[struct={r['structural']:.2f} loc={r['location']:.2f} fac={r['facilities']:.2f}]  {sub}")
    print()

QUERY: DLF The Arbour  |  4 BHK Apartment in Sector 63 Gurgaon
  1. DLF The Summit                   score=0.553  [struct=0.99 loc=0.09 fac=0.16]  4 BHK Apartment in Sector 54, Gurgaon
  2. DLF The Pinnacle                 score=0.528  [struct=1.00 loc=0.08 fac=0.01]  4 BHK Apartment in DLF Phase 5, Gurgaon
  3. Tulip Purple                     score=0.528  [struct=0.85 loc=0.10 fac=0.35]  4 BHK Apartment in Sector 69, Gurgaon
  4. Paras Quartier                   score=0.526  [struct=0.98 loc=0.10 fac=0.02]  4 BHK Apartment in Gwal Pahari, Gurgaon
  5. BPTP Mansions Park Prime         score=0.518  [struct=0.87 loc=0.09 fac=0.29]  4 BHK Apartment in Sector 66, Gurgaon

QUERY: M3M Golf Hills  |  2, 3, 4 BHK Apartment in Sector 79, Gurgaon
  1. Corona Optus                     score=0.555  [struct=0.95 loc=0.09 fac=0.25]  2, 3, 4 BHK Apartment in Sector 37C, Gurgaon
  2. Puri Emerald Bay                 score=0.552  [struct=0.95 loc=0.09 fac=0.24]  2, 3, 4 BHK Apartment in Sector 104, Gu

## 3. Weight sensitivity

The chosen weights should not be knife-edge, and should beat both equal-weighting
and the original notebook's ratio.

In [4]:
ALTS = {
    "0.5/0.3/0.2 (chosen)":   {"structural": .5,   "location": .3,   "facilities": .2},
    "0.4/0.4/0.2":            {"structural": .4,   "location": .4,   "facilities": .2},
    "0.6/0.2/0.2":            {"structural": .6,   "location": .2,   "facilities": .2},
    "equal (1/3 each)":       {"structural": 1/3,  "location": 1/3,  "facilities": 1/3},
    "notebook 6/5/3 -> norm": {"structural": 5/14, "location": 3/14, "facilities": 6/14},
}
for label, w in ALTS.items():
    r = SimilarProjectsRecommender(raw, weights=w)
    print(label)
    for t in TARGETS:
        print(f"  {t:<21} -> {r.recommend(t, k=5)['PropertyName'].tolist()}")
    print()

0.5/0.3/0.2 (chosen)
  DLF The Arbour        -> ['DLF The Summit', 'DLF The Pinnacle', 'Tulip Purple', 'Paras Quartier', 'BPTP Mansions Park Prime']
  M3M Golf Hills        -> ['Corona Optus', 'Puri Emerald Bay', 'Unitech Escape', 'BPTP Terra', 'Unitech Harmony']
  Ireo Victory Valley   -> ['Ambience Creacions', 'Pioneer Urban Presidia', 'Pioneer Araya', 'Bestech Park View Grand Spa', 'DLF The Crest']

0.4/0.4/0.2
  DLF The Arbour        -> ['DLF The Summit', 'Tulip Purple', 'BPTP Mansions Park Prime', 'Paras Quartier', 'DLF The Pinnacle']
  M3M Golf Hills        -> ['Corona Optus', 'Puri Emerald Bay', 'Unitech Escape', 'BPTP Terra', 'Unitech Harmony']
  Ireo Victory Valley   -> ['Ambience Creacions', 'Pioneer Urban Presidia', 'Bestech Park View Grand Spa', 'Pioneer Araya', 'DLF The Crest']



0.6/0.2/0.2
  DLF The Arbour        -> ['DLF The Summit', 'DLF The Pinnacle', 'Paras Quartier', 'Tulip Purple', 'BPTP Mansions Park Prime']
  M3M Golf Hills        -> ['Corona Optus', 'Puri Emerald Bay', 'Unitech Escape', 'BPTP Terra', 'Unitech Harmony']
  Ireo Victory Valley   -> ['Ambience Creacions', 'Pioneer Urban Presidia', 'Pioneer Araya', 'Bestech Park View Grand Spa', 'DLF The Crest']



equal (1/3 each)
  DLF The Arbour        -> ['Tulip Purple', 'JMS The Nation', 'Oxirich Chintamanis', 'Vatika Aspiration', 'BPTP Mansions Park Prime']
  M3M Golf Hills        -> ['Ashiana Amarah', 'Corona Optus', 'Puri Emerald Bay', 'Mahindra Aura', 'Unitech Escape']
  Ireo Victory Valley   -> ['Pioneer Urban Presidia', 'Ambience Creacions', 'Silverglades The Melia', 'DLF The Crest', 'AIPL The Peaceful Homes']

notebook 6/5/3 -> norm
  DLF The Arbour        -> ['JMS The Nation', 'Oxirich Chintamanis', 'Vatika Aspiration', 'Tulip Purple', 'SS Linden Floors']
  M3M Golf Hills        -> ['Ashiana Amarah', 'Corona Optus', 'Puri Emerald Bay', 'Godrej Nature Plus Serenity', 'Birla Navya Avik']
  Ireo Victory Valley   -> ['Pioneer Urban Presidia', 'Silverglades The Melia', 'DLF The Crest', 'Ambience Creacions', 'AIPL The Peaceful Homes']



## Takeaway

- Blend = `structural 0.5 / location 0.3 / facilities 0.2` on min-max-normalised
  matrices. Reasoning, scale table, and known limitations:
  `reports/recommender/blend_weights.md`.
- `structural` (BHK config / area / price band) carries the ranking; `facilities`
  is a tie-breaker; `location` scored above 0.2 in only 1 of the 15
  sanity-check recommendations — at effective weight it is closer to a rare
  tiebreak than a 30 % input, kept for when it does fire and for when a denser
  location signal (`sector`) replaces the sparse landmark distances.
- Listing-level preference matching is out of scope here — `PROJECT_PLAN.md` §10.
